# Hyperparameter Selection: Individual Storms

Notebook to explore the optimal set of clustering hyperparameters, by looking at aggregate plots from papers, as well as tracking known storms.

Jimmy Butler, April 2025

There are several hyperparameters we can vary:
+ $\epsilon_{space}$
+ $\epsilon_{time}$
+ min_pts
+ rep_pts

We will explore different combinations of these hyperparameters, and see how good/bad these choices are at detecting important extreme events.

Defaults are the following, and in the below, the parameter value listed indicates the one which was changed.
+ $\epsilon_{space} = 500$ km
+ $\epsilon_{time} = 12$ hr
+ min_pts: 5
+ rep_pts: 10

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import os

import matplotlib.path as mpath
import matplotlib.colors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

from matplotlib import animation
import matplotlib
matplotlib.use("Agg")
from ipywidgets import Video
from matplotlib.cm import prism
plt.rcParams['animation.ffmpeg_path'] = '/usr/bin/ffmpeg'

from ipywidgets import IntProgress
from IPython.display import display

repo_dir = str(Path(os.getcwd()).parents[1])

os.chdir(repo_dir + '/scripts/')
from utils import display_catalog
import utils

ani_dir = repo_dir + '/animations/hyperparam_selection/'
data_dir = repo_dir + '/data/hyperparameter_individual_storms/'

In [2]:
def average_angle(subdf):
    lats = np.radians(subdf.lat)
    lons = np.radians(subdf.lon)

    x = np.cos(lats)*np.cos(lons)
    y = np.cos(lats)*np.sin(lons)
    z = np.sin(lats)
    
    avg_x = np.mean(x)
    avg_y = np.mean(y)
    avg_z = np.mean(z)

    avg_lat = np.arcsin(avg_z)
    avg_lon = arctan(avg_x, avg_y)

    return pd.Series({'avg_lat':np.degrees(avg_lat), 'avg_lon':np.degrees(avg_lon)})
    
def arctan(x, y):
    if y/x > 0:
        if x > 0:
            return(np.arctan(y/x))
        else:
            return(np.arctan(y/x)-np.pi)
    else:
        if x > 0:
            return(np.arctan(y/x))
        else:
            return(np.pi+np.arctan(y/x))

In [3]:
def compute_duration(ar_da):
    days = (ar_da.time.max() - ar_da.time.min()).values.astype('timedelta64[h]').astype(int) + np.timedelta64(3, 'h')
    return days

def add_start_date(ar_da):
    start = ar_da.time.min().values
    return start

def add_end_date(ar_da):
    end = ar_da.time.max().values
    return end

In [4]:
def make_movie(stormtime_df, title):
    '''
    Function that constructs an animation of ARs in a stormtime version of the AR catalog.

    Inputs
        stormtime_df (pandas DataFrame): same as above
        title (string): the title to print on the animation

    Outputs
        ani (Matplotlib Animation): an animation object, as far as I know it must be saved first in order to view it
    '''

    # define the times of the movie, and mappings between AR labels and colors
    movie_times = pd.date_range(start=stormtime_df.time.min(), end=stormtime_df.time.max(), freq='3h')
    unique_clusters = stormtime_df['label'].unique()
    color_mapping = {unique_clusters[j]:prism(j/len(unique_clusters)) for j in range(len(unique_clusters)) }

    # plot the jth frame of the movie (need as input to matplotlib animation constructor)
    def plt_time(j):
        time_pt = movie_times[j]

        if (time_pt == stormtime_df.time).any():
            dat = stormtime_df[stormtime_df['time'] == time_pt]
            n_clusts = dat.shape[0]

            for i in range(n_clusts):
                cluster = dat['label'].iloc[i]
                ax.scatter(dat['lon'].iloc[i], dat['lat'].iloc[i], transform=ccrs.PlateCarree(), s=1, color=color_mapping[cluster], label=str(cluster), zorder=30)
                ax.text(dat['avg_lon'].iloc[i], dat['avg_lat'].iloc[i], cluster, transform=ccrs.PlateCarree(), fontsize=16, zorder=35)

        ax.set_extent([-180,180,-90,-39], ccrs.PlateCarree())
        ice_shelf_poly = cfeature.NaturalEarthFeature('physical', 'antarctic_ice_shelves_polys', '50m',edgecolor='none',facecolor='lightcyan') # 10m, 50m, 110m
        ax.add_feature(ice_shelf_poly,linewidth=3)
        ice_shelf_line = cfeature.NaturalEarthFeature('physical', 'antarctic_ice_shelves_lines', '50m',edgecolor='black',facecolor='none') # 10m, 50m, 110m
        ax.add_feature(ice_shelf_line,linewidth=1,zorder=13)
        ax.coastlines(resolution='110m',linewidth=1,zorder=32)

    
        # Map extent 
        theta = np.linspace(0, 2*np.pi, 100)
        center, radius = [0.5, 0.5], 0.5
        verts = np.vstack([np.sin(theta), np.cos(theta)]).T
        circle = mpath.Path(verts * radius + center)
        ax.set_boundary(circle, transform=ax.transAxes)
        ax.gridlines(alpha=0.5, zorder=33)
    
        time_ts = pd.Timestamp(time_pt)
        ax.set_title(f'{time_ts.month}/{time_ts.day}/{time_ts.year}, {time_ts.hour}:00')

    # instantiate the animation
    fig, ax = plt.subplots(figsize=(5,5), subplot_kw=dict(projection=ccrs.Stereographic(central_longitude=0., central_latitude=-90.)))
    plt_time(0)
    fig.suptitle(title, fontsize=16)

    def update_img(i):
        ax.clear()
        plt_time(i)

    ani = animation.FuncAnimation(fig, update_img, frames=len(movie_times))

    return ani

def save_progress(i, n):
    '''
    Function that will be used in calls to animation.save(), just to give some progress updates as it can take a while to save!
    '''
    if i%20 == 0: {print(f'Saving frame {i}/{n}')}

## Default Specifications

In [5]:
hyperparam_dir = 'space500_time12_minpts5_reppts10/'

### Individual Storms

#### Storm 1: March 23-24, 2015 (Bozkurt et al.)

In [7]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '03_2324_15.h5')
catalog_subset = catalog_subset.set_index(catalog_subset.index.astype('int'))
stormtime_df = utils.to_stormtime_format(catalog_subset)
stormtime_df = pd.concat([stormtime_df, stormtime_df.apply(average_angle, axis=1)], axis=1)
ani = make_movie(stormtime_df, 'March 23-24, 2015 (Bozkurt 2015)')
ani.save(ani_dir + hyperparam_dir + '03_2324_15.gif', progress_callback=save_progress)

MovieWriter ffmpeg unavailable; using Pillow instead.


Saving frame 0/49
Saving frame 20/49
Saving frame 40/49


#### Storm 2: January 8-10, 2019 (Gehring 2022)

In [9]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '01_0810_19.h5')
catalog_subset = catalog_subset.set_index(catalog_subset.index.astype('int'))
stormtime_df = utils.to_stormtime_format(catalog_subset)
stormtime_df = pd.concat([stormtime_df, stormtime_df.apply(average_angle, axis=1)], axis=1)
ani = make_movie(stormtime_df, 'January 8-10, 2019 (Gehring 2022)')
ani.save(ani_dir + hyperparam_dir + '01_0810_19.gif', progress_callback=save_progress)

MovieWriter ffmpeg unavailable; using Pillow instead.


Saving frame 0/40
Saving frame 20/40


#### Storm 3: February 7-9, 2022 (Gorodetskaya 2023)

In [10]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '02_0409_22.h5')
catalog_subset = catalog_subset.set_index(catalog_subset.index.astype('int'))
stormtime_df = utils.to_stormtime_format(catalog_subset)
stormtime_df = pd.concat([stormtime_df, stormtime_df.apply(average_angle, axis=1)], axis=1)
ani = make_movie(stormtime_df, 'February 7-9, 2022 (Gorodetskaya 2023)')
ani.save(ani_dir + hyperparam_dir + '02_0409_22.gif', progress_callback=save_progress)

MovieWriter ffmpeg unavailable; using Pillow instead.


Saving frame 0/34
Saving frame 20/34


#### Storm 4: November 16-17, 2018 (Gorodetskaya 2020)

In [11]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '11_1617_18.h5')
catalog_subset = catalog_subset.set_index(catalog_subset.index.astype('int'))
stormtime_df = utils.to_stormtime_format(catalog_subset)
stormtime_df = pd.concat([stormtime_df, stormtime_df.apply(average_angle, axis=1)], axis=1)
ani = make_movie(stormtime_df, 'November 16-17, 2018 (Gorodetskaya 2020)')
ani.save(ani_dir + hyperparam_dir + '11_1617_18.gif', progress_callback=save_progress)

MovieWriter ffmpeg unavailable; using Pillow instead.


Saving frame 0/40
Saving frame 20/40


#### Storm 5: December 18, 2018 (Gorodetskaya 2020)

In [12]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '12_1818_18.h5')
catalog_subset = catalog_subset.set_index(catalog_subset.index.astype('int'))
stormtime_df = utils.to_stormtime_format(catalog_subset)
stormtime_df = pd.concat([stormtime_df, stormtime_df.apply(average_angle, axis=1)], axis=1)
ani = make_movie(stormtime_df, 'December 18, 2018 (Gorodetskaya 2020)')
ani.save(ani_dir + hyperparam_dir + '12_1818_18.gif', progress_callback=save_progress)

MovieWriter ffmpeg unavailable; using Pillow instead.


Saving frame 0/17


#### Storm 6-8 (family event): (1) February 2, 2020; (2) February 3-5, 2020; (3) February 7-8, 2020 (Maclennan 2023)

In [13]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '02_family_20.h5')
catalog_subset = catalog_subset.set_index(catalog_subset.index.astype('int'))
stormtime_df = utils.to_stormtime_format(catalog_subset)
stormtime_df = pd.concat([stormtime_df, stormtime_df.apply(average_angle, axis=1)], axis=1)
ani = make_movie(stormtime_df, 'February 2020 AR Family (Maclennan 2023)')
ani.save(ani_dir + hyperparam_dir + '02_family_20.gif', progress_callback=save_progress)

MovieWriter ffmpeg unavailable; using Pillow instead.


Saving frame 0/106
Saving frame 20/106
Saving frame 40/106
Saving frame 60/106
Saving frame 80/106
Saving frame 100/106


#### Storm 9: March 14-18, 2022 (Wille 2024)

In [14]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '03_1418_22.h5')
catalog_subset = catalog_subset.set_index(catalog_subset.index.astype('int'))
stormtime_df = utils.to_stormtime_format(catalog_subset)
stormtime_df = pd.concat([stormtime_df, stormtime_df.apply(average_angle, axis=1)], axis=1)
ani = make_movie(stormtime_df, 'March 14-18, 2022 (Wille 2024)')
ani.save(ani_dir + hyperparam_dir + '03_1418_22.gif', progress_callback=save_progress)

MovieWriter ffmpeg unavailable; using Pillow instead.


Saving frame 0/50
Saving frame 20/50
Saving frame 40/50


#### Storm 10: January 24-26, 2008 (Wille 2022)

In [15]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '01_2426_08.h5')
catalog_subset = catalog_subset.set_index(catalog_subset.index.astype('int'))
stormtime_df = utils.to_stormtime_format(catalog_subset)
stormtime_df = pd.concat([stormtime_df, stormtime_df.apply(average_angle, axis=1)], axis=1)
ani = make_movie(stormtime_df, 'January 24-26, 2008 (Wille 2024)')
ani.save(ani_dir + hyperparam_dir + '01_2426_08.gif', progress_callback=save_progress)

MovieWriter ffmpeg unavailable; using Pillow instead.


Saving frame 0/31
Saving frame 20/31


### Aggregate Statistics/Plots

## Rep Pts Variations

### rep_pts = $20$

In [4]:
hyperparam_dir = 'space500_time12_minpts5_reppts20/'

#### Individual Storms

#### Storm 1: March 23-24, 2015 (Bozkurt et al.)

In [15]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '03_2324_15.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'March 23-24, 2015 (Bozkurt 2015)')
ani.save(ani_dir + hyperparam_dir + '03_2324_15.mp4', progress_callback=save_progress)

Saving frame 0/49
Saving frame 20/49
Saving frame 40/49


In [16]:
Video.from_file(ani_dir + hyperparam_dir + '03_2324_15.mp4',)

Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 2: January 8-10, 2019 (Gehring 2022)

In [17]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '01_0810_19.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'January 8-10, 2019 (Gehring 2022)')
ani.save(ani_dir + hyperparam_dir + '01_0810_19.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '01_0810_19.mp4',)

Saving frame 0/40
Saving frame 20/40


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 3: February 7-9, 2022 (Gorodetskaya 2023)

In [12]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '02_0409_22.h5')
catalog_subset = catalog_subset.set_index(catalog_subset.index.astype('int'))
stormtime_df = utils.to_stormtime_format(catalog_subset)
stormtime_df = pd.concat([stormtime_df, stormtime_df.apply(average_angle, axis=1)], axis=1)

ani = make_movie(stormtime_df, 'February 7-9, 2022 (Gorodetskaya 2023)')
ani.save(ani_dir + hyperparam_dir + '02_0409_22.gif', progress_callback=save_progress)

MovieWriter ffmpeg unavailable; using Pillow instead.


Saving frame 0/34
Saving frame 20/34


#### Storm 4: November 16-17, 2018 (Gorodetskaya 2020)

In [5]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '11_1617_18.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'November 16-17, 2018 (Gorodetskaya 2020)')
ani.save(ani_dir + hyperparam_dir + '11_1617_18.mp4', progress_callback=save_progress)


KeyError: 'avg_lon'

#### Storm 5: December 18, 2018 (Gorodetskaya 2020)

In [20]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '12_1818_18.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'December 18, 2018 (Gorodetskaya 2020)')
ani.save(ani_dir + hyperparam_dir + '12_1818_18.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '12_1818_18.mp4',)

Saving frame 0/17


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 6-8 (family event): (1) February 2, 2020; (2) February 3-5, 2020; (3) February 7-8, 2020 (Maclennan 2023)

In [21]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '02_family_20.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'February 2020 AR Family (Maclennan 2023)')
ani.save(ani_dir + hyperparam_dir + '02_family_20.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '02_family_20.mp4',)

Saving frame 0/106
Saving frame 20/106
Saving frame 40/106
Saving frame 60/106
Saving frame 80/106
Saving frame 100/106


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 9: March 14-18, 2022 (Wille 2024)

In [13]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '03_1418_22.h5')
catalog_subset = catalog_subset.set_index(catalog_subset.index.astype('int'))
stormtime_df = utils.to_stormtime_format(catalog_subset)
stormtime_df = pd.concat([stormtime_df, stormtime_df.apply(average_angle, axis=1)], axis=1)

ani = make_movie(stormtime_df, 'March 14-18, 2022 (Wille 2024)')
ani.save(ani_dir + hyperparam_dir + '03_1418_22.gif', progress_callback=save_progress)

MovieWriter ffmpeg unavailable; using Pillow instead.


Saving frame 0/50
Saving frame 20/50
Saving frame 40/50


In [22]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '03_1418_22.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'March 14-18, 2022 (Wille 2024)')
ani.save(ani_dir + hyperparam_dir + '03_1418_22.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '03_1418_22.mp4',)

Saving frame 0/50
Saving frame 20/50
Saving frame 40/50


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 10: January 24-26, 2008 (Wille 2022)

In [23]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '01_2426_08.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'January 24-26, 2008 (Wille 2024)')
ani.save(ani_dir + hyperparam_dir + '01_2426_08.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '01_2426_08.mp4',)

Saving frame 0/31
Saving frame 20/31


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Aggregate Statistics/Plots

## Time Epsilon Variations

### $\epsilon_{time}=16$

In [24]:
hyperparam_dir = 'space500_time16_minpts5_reppts10/'

#### Individual Storms

#### Storm 1: March 23-24, 2015 (Bozkurt et al.)

In [25]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '03_2324_15.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'March 23-24, 2015 (Bozkurt 2015)')
ani.save(ani_dir + hyperparam_dir + '03_2324_15.mp4', progress_callback=save_progress)

Saving frame 0/49
Saving frame 20/49
Saving frame 40/49


In [26]:
Video.from_file(ani_dir + hyperparam_dir + '03_2324_15.mp4',)

Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 2: January 8-10, 2019 (Gehring 2022)

In [27]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '01_0810_19.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'January 8-10, 2019 (Gehring 2022)')
ani.save(ani_dir + hyperparam_dir + '01_0810_19.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '01_0810_19.mp4',)

Saving frame 0/40
Saving frame 20/40


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 3: February 7-9, 2022 (Gorodetskaya 2023)

In [28]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '02_0409_22.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'February 7-9, 2022 (Gorodetskaya 2023)')
ani.save(ani_dir + hyperparam_dir + '02_0409_22.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '02_0409_22.mp4',)

Saving frame 0/34
Saving frame 20/34


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 4: November 16-17, 2018 (Gorodetskaya 2020)

In [29]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '11_1617_18.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'November 16-17, 2018 (Gorodetskaya 2020)')
ani.save(ani_dir + hyperparam_dir + '11_1617_18.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '11_1617_18.mp4',)

Saving frame 0/40
Saving frame 20/40


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 5: December 18, 2018 (Gorodetskaya 2020)

In [30]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '12_1818_18.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'December 18, 2018 (Gorodetskaya 2020)')
ani.save(ani_dir + hyperparam_dir + '12_1818_18.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '12_1818_18.mp4',)

/tmp/ipykernel_181/3995199149.py:52: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(5,5), subplot_kw=dict(projection=ccrs.Stereographic(central_longitude=0., central_latitude=-90.)))


Saving frame 0/17


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 6-8 (family event): (1) February 2, 2020; (2) February 3-5, 2020; (3) February 7-8, 2020 (Maclennan 2023)

In [31]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '02_family_20.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'February 2020 AR Family (Maclennan 2023)')
ani.save(ani_dir + hyperparam_dir + '02_family_20.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '02_family_20.mp4',)

Saving frame 0/106
Saving frame 20/106
Saving frame 40/106
Saving frame 60/106
Saving frame 80/106
Saving frame 100/106


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 9: March 14-18, 2022 (Wille 2024)

In [32]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '03_1418_22.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'March 14-18, 2022 (Wille 2024)')
ani.save(ani_dir + hyperparam_dir + '03_1418_22.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '03_1418_22.mp4',)

Saving frame 0/50
Saving frame 20/50
Saving frame 40/50


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 10: January 24-26, 2008 (Wille 2022)

In [33]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '01_2426_08.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'January 24-26, 2008 (Wille 2024)')
ani.save(ani_dir + hyperparam_dir + '01_2426_08.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '01_2426_08.mp4',)

Saving frame 0/31
Saving frame 20/31


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

### $\epsilon_{time}=18$

In [34]:
hyperparam_dir = 'space500_time18_minpts5_reppts10/'

#### Storm 1: March 23-24, 2015 (Bozkurt et al.)

In [35]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '03_2324_15.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'March 23-24, 2015 (Bozkurt 2015)')
ani.save(ani_dir + hyperparam_dir + '03_2324_15.mp4', progress_callback=save_progress)

Saving frame 0/49
Saving frame 20/49
Saving frame 40/49


In [36]:
Video.from_file(ani_dir + hyperparam_dir + '03_2324_15.mp4',)

Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 2: January 8-10, 2019 (Gehring 2022)

In [37]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '01_0810_19.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'January 8-10, 2019 (Gehring 2022)')
ani.save(ani_dir + hyperparam_dir + '01_0810_19.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '01_0810_19.mp4',)

Saving frame 0/40
Saving frame 20/40


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 3: February 7-9, 2022 (Gorodetskaya 2023)

In [38]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '02_0409_22.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'February 7-9, 2022 (Gorodetskaya 2023)')
ani.save(ani_dir + hyperparam_dir + '02_0409_22.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '02_0409_22.mp4',)

Saving frame 0/34
Saving frame 20/34


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 4: November 16-17, 2018 (Gorodetskaya 2020)

In [39]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '11_1617_18.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'November 16-17, 2018 (Gorodetskaya 2020)')
ani.save(ani_dir + hyperparam_dir + '11_1617_18.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '11_1617_18.mp4',)

Saving frame 0/40
Saving frame 20/40


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 5: December 18, 2018 (Gorodetskaya 2020)

In [40]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '12_1818_18.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'December 18, 2018 (Gorodetskaya 2020)')
ani.save(ani_dir + hyperparam_dir + '12_1818_18.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '12_1818_18.mp4',)

Saving frame 0/17


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 6-8 (family event): (1) February 2, 2020; (2) February 3-5, 2020; (3) February 7-8, 2020 (Maclennan 2023)

In [41]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '02_family_20.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'February 2020 AR Family (Maclennan 2023)')
ani.save(ani_dir + hyperparam_dir + '02_family_20.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '02_family_20.mp4',)

Saving frame 0/106
Saving frame 20/106
Saving frame 40/106
Saving frame 60/106
Saving frame 80/106
Saving frame 100/106


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 9: March 14-18, 2022 (Wille 2024)

In [42]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '03_1418_22.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'March 14-18, 2022 (Wille 2024)')
ani.save(ani_dir + hyperparam_dir + '03_1418_22.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '03_1418_22.mp4',)

Saving frame 0/50
Saving frame 20/50
Saving frame 40/50


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Storm 10: January 24-26, 2008 (Wille 2022)

In [43]:
catalog_subset = pd.read_hdf(data_dir + hyperparam_dir + '01_2426_08.h5')
stormtime_df = utils.to_stormtime_format(catalog_subset)
ani = make_movie(stormtime_df, 'January 24-26, 2008 (Wille 2024)')
ani.save(ani_dir + hyperparam_dir + '01_2426_08.mp4', progress_callback=save_progress)
stormtime_df = utils.to_stormtime_format(catalog_subset)
Video.from_file(ani_dir + hyperparam_dir + '01_2426_08.mp4',)

Saving frame 0/31
Saving frame 20/31


Video(value=b'\x00\x00\x00 ftypisom\x00\x00\x02\x00isomiso2avc1mp41\x00\x00\x00\x08free...')

#### Individual Storms